# 1. 环境配置
## 1.1 安装 openai 库

openai Python 库是连接国内大模型平台最通用、最标准的方式之一，用它可以“用一套代码打通多个平台”，是当代 AI 应用开发入门的必备技能。

In [1]:
! pip install openai==2.11.0 dashscope==1.25.4

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
  Using cached https://pypi.tuna.tsinghua.edu.cn/packages/e5/f1/d9251b565fce9f8daeb45611e3e0d2f7f248429e40908dcee3b6fe1b5944/openai-2.11.0-py3-none-any.whl (1.1 MB)
  Using cached https://pypi.tuna.tsinghua.edu.cn/packages/a5/33/e9e0f4663e55f72be6408f294a42ef15da3095b8bd014275502e88085e4c/dashscope-1.25.4-py3-none-any.whl (1.3 MB)
  Using cached https://pypi.tuna.tsinghua.edu.cn/packages/38/0e/27be9fdef66e72d64c0cdc3cc2823101b80585f8119b5c112c2e8f5f7dab/anyio-4.12.1-py3-none-any.whl (113 kB)
  Using cached https://pypi.tuna.tsinghua.edu.cn/packages/12/b3/231ffd4ab1fc9d679809f356cebee130ac7daa00d6d6f3206dd4fd137e9e/distro-1.9.0-py3-none-any.whl (20 kB)
  Using cached https://pypi.tuna.tsinghua.edu.cn/packages/2a/39/e50c7c3a983047577ee07d2a9e53faf5a69493943ec3f6a384bdc792deb2/httpx-0.28.1-py3-none-any.whl (73 kB)
  Using cached https://pypi.tuna.tsinghua.edu.cn/packages/1c/c7/6659f537f9562d963488e3e55573498a442503ced01f7e169e96

## 1.2 获取大模型 API_Key

### 1.2.1 AI Studio 
在 AI Studio 的 BML 中已经内置了默认的 API_Key, 因此我们无需修改即可进行使用。假如我们希望在本地调用，可以在[我的控制台](https://aistudio.baidu.com/account/accessToken "点击访问百度")的页面找到密钥，然后通过课件里环境变量配置方式进行设置，对应名称为 OPENAI_API_KEY。

In [2]:
import os

# 如未使用环境变量配置 API Key，可取消以下注释并填写你的 Key
# os.environ["OPENAI_API_KEY"] = "sk-xxxxxxxx"

### 1.2.2 百炼大模型平台
请先前往[阿里云百炼大模型平台](https://bailian.console.aliyun.com/?tab=model#/api-key)注册账号，然后通过同样的方式设置为 DASHSCOPE_API_KEY 即可。

In [3]:
import os

# 如未使用环境变量配置 API Key，可取消以下注释并填写你的 Key
# os.environ["DASHSCOPE_API_KEY"] = "sk-yyyyyyyy"

## 1.3 获取 AI Studio 模型信息

我们可以通过以下代码来了解 AI Studio 中所支持的所有模型。另外我们也可以通过[百度大模型 API 文档](https://www.baidu.com "点击访问百度")获取更多相关信息。

In [4]:
import os
from openai import OpenAI

client = OpenAI(
    api_key=os.environ.get("OPENAI_API_KEY"),  # 含有 AI Studio 访问令牌的环境变量，https://aistudio.baidu.com/account/accessToken,
    base_url="https://aistudio.baidu.com/llm/lmapi/v3",  # aistudio 大模型 api 服务域名
)

models = client.models.list()
for model in models.data:
    print(model.id)

embedding-v1
Stable-Diffusion-XL
ernie-3.5-8k
ernie-4.0-8k
ernie-4.0-turbo-8k
ernie-speed-8k
ernie-speed-128k
ernie-tiny-8k
ernie-char-8k
ernie-lite-8k
bge-large-zh
ernie-4.0-turbo-128k
deepseek-r1
deepseek-v3
ernie-4.5-turbo-vl-32k
ernie-4.5-turbo-128k
ernie-4.5-turbo-32k
ernie-x1-turbo-32k
deepseek-r1-250528
ernie-lite-pro-128k
ernie-speed-pro-128k
ernie-4.0-turbo-8k-latest
ernie-4.0-8k-latest
qwq-32b
qwen2.5-vl-7b-instruct
qwen2.5-vl-32b-instruct
qwen2.5-7b-instruct
llama-4-scout-17b-16e-instruct
llama-4-maverick-17b-128e-instruct
qwen3-4b
qwen3-8b
qwen3-32b
qwen3-30b-a3b
qwen3-235b-a22b
ernie-4.5-vl-28b-a3b
ernie-4.5-21b-a3b
ernie-4.5-0.3b
ernie-4.5-turbo-vl-preview
ernie-4.5-turbo-128k-preview
qwen3-coder-30b-a3b-instruct
kimi-k2-instruct
qwen3-coder-480b-a35b-instruct
ernie-4.5-turbo-vl
ernie-x1.1-preview
ernie-4.5-21b-a3b-thinking
ernie-4.5-vl-28b-a3b-thinking
ernie-5.0-thinking-preview


# 2. 模型调用

在使用大模型 API 时，我们通常会遇到两种输出模式：普通输出（非流式） 和 流式输出（Stream）。

## 2.1 非流式输出

特点：
- 一次性返回完整的模型生成结果。
- 代码逻辑简单，适合短文本或对实时性要求不高的场景。

执行流程：
1. 发送请求给模型。
2. 模型在服务器端完成推理，生成完整结果。
3. 一次性返回完整回答，供后续处理。

适用场景：

- 短文本问答
- 批量任务处理
- 对实时性要求不高的情况

In [5]:
import os
from openai import OpenAI

# 1. 创建 OpenAI 客户端实例
# 这里我们使用 OpenAI 提供的 SDK，但指定了自定义的 base_url
# 因为百度 AI Studio 提供了兼容 OpenAI API 规范的接口
# 所以只要替换 base_url 和模型名称就可以调用百度的模型
client = OpenAI(
    api_key=os.environ.get("OPENAI_API_KEY"),  # 从系统环境变量中读取 API Key，避免在代码中写死，保证安全
    base_url="https://aistudio.baidu.com/llm/lmapi/v3",  # 指定 API 接口的基础 URL，这里是百度 AI Studio 的接口地址
)

# 2. 调用 Chat Completion 接口，发起一次对话请求
chat_completion = client.chat.completions.create(
    messages=[  # 对话的历史消息，支持多轮对话
        {
            'role': 'system',  # 系统角色，用于设定 AI 助手的身份和行为
            'content': '你是 AI Studio 实训AI开发平台的开发者助理，你精通开发相关的知识，负责给开发者提供搜索帮助建议。'
        },
        {
            'role': 'user',  # 用户角色，表示这是用户输入的内容
            'content': '你好，请介绍一下AI Studio'
        }
    ],
    model="ernie-3.5-8k",  # 指定调用的模型，这里是百度文心大模型的 3.5 版本，支持 8k 上下文
)

# 3. 打印 AI 模型的回复
# choices[0] 表示取第一个回答（有时可能返回多个回答）
# message.content 表示提取回答的具体文本内容
print(chat_completion.choices[0].message.content)

您好！AI Studio 是百度推出的一个**一站式AI开发实训平台**，专注于为开发者提供便捷、高效的AI项目开发环境与学习资源。以下是AI Studio的核心功能与特点介绍：

### 1. **核心功能**
   - **在线开发环境**：支持Jupyter Notebook、PyCharm等开发工具，提供免费GPU算力（如V100、A100），无需本地配置即可运行深度学习模型。
   - **丰富数据集与模型库**：内置海量公开数据集（如ImageNet、COCO）和预训练模型（如PaddlePaddle生态模型），支持快速调用与二次开发。
   - **实训项目与课程**：提供从入门到进阶的AI教程（如计算机视觉、NLP、强化学习），结合实战案例（如人脸识别、目标检测）帮助快速上手。
   - **竞赛与社区**：定期举办AI竞赛（如Kaggle风格挑战），开发者可提交作品参与排名；社区支持问题交流、代码分享与协作。

### 2. **技术栈支持**
   - **深度学习框架**：以百度自研的**PaddlePaddle**为核心，同时兼容TensorFlow、PyTorch等主流框架。
   - **自动化工具**：集成PaddleHub（模型库）、PaddleNLP（自然语言处理工具包）、PaddleDetection（目标检测框架）等，简化开发流程。
   - **部署能力**：支持模型一键导出为ONNX格式，或通过Paddle Inference、Paddle Serving快速部署到云端/边缘设备。

### 3. **适用场景**
   - **学习与实训**：学生或初学者通过项目案例掌握AI技能。
   - **科研与实验**：研究人员利用免费算力快速验证算法。
   - **企业应用**：开发者基于预训练模型快速构建AI应用（如智能客服、图像分类）。

### 4. **优势特点**
   - **零成本入门**：免费算力+开源资源，降低AI开发门槛。
   - **全流程覆盖**：从数据准备、模型训练到部署的一站式服务。
   - **生态支持**：背靠百度AI技术，提供产业级解决方案与案例。

### 5. **典型案例**
   - 开发者通过AI Studio完成**口罩检测模型**训练，并部署到摄像头实现实时识别。
   - 

## 2.2 流式输出（Stream）

特点：

- 模型生成内容的同时分块（chunk）返回。
- 用户可以像“实时打字”一样看到逐步生成的内容。
- 适合长文本或实时交互场景。

执行流程：

1. 开启 stream=True 参数。
2. 模型生成内容时分多次返回，每次返回一个数据块。
3. 客户端逐块处理输出，实时展示。

适用场景：

- 长文本生成（如文章、代码）
- 实时交互（如聊天机器人）
- 对用户体验要求高的前端应用

In [6]:
import os
from openai import OpenAI

# 1. 初始化 OpenAI 客户端
# - 通过 OpenAI 提供的 SDK 创建客户端实例
# - 这里指定了 API Key（从环境变量中读取，保证安全性）
# - 指定了 base_url，指向百度 AI Studio 兼容 OpenAI API 的接口地址
client = OpenAI(
    api_key=os.environ.get("OPENAI_API_KEY"),  # 从系统环境变量中读取 API Key，建议在系统中提前设置
    base_url="https://aistudio.baidu.com/llm/lmapi/v3",  # 百度 AI Studio 提供的 API 入口
)

# 2. 创建一个 **流式对话（stream=True）**
# - 这里调用 chat.completions.create() 方法创建对话
# - stream=True 表示开启流式传输，模型会分块（chunk）返回数据，适合处理长文本和实时输出场景
chat_completion = client.chat.completions.create(
    model="ernie-3.5-8k",  # 指定使用的模型，这里是文心一言 3.5 模型，支持 8k token 上下文
    messages=[  # 定义对话历史，支持多轮对话
        {
            "role": "system",  # 系统角色：用于设定 AI 的身份、知识领域或行为风格
            "content": "你是 AI Studio 实训AI开发平台的开发者助理，你精通开发相关的知识，负责给开发者提供搜索帮助建议。"
        },
        {
            "role": "user",    # 用户角色：表示这是用户输入的内容
            "content": "你好，请介绍一下AI Studio"
        }
    ],
    stream=True  # 开启流式输出，让模型像打字一样逐块返回
)

# 3. 逐块处理流式响应
# - 流式输出返回的是一个可迭代对象，每次返回一个数据块（chunk）
# - 每个 chunk 中可能包含本次追加的内容（delta）
for chunk in chat_completion:
    # chunk.choices：返回的候选回答列表
    # chunk.choices[0].delta.content：本次追加的文本内容（可能为空，需要判断）
    if chunk.choices and chunk.choices[0].delta.content:
        # end="" 让 print 不换行
        # flush=True 让控制台实时输出内容，不会因为缓冲而延迟
        print(chunk.choices[0].delta.content, end="", flush=True)

# 4. 在流式输出完成后，统一换行，避免最后一行和后续输出粘在一起
print()


您好！AI Studio 是百度推出的一个集成了算法开发、模型训练、模型部署等一站式服务的AI开发平台。它为开发者提供了丰富的工具和资源，帮助您更高效地进行AI项目的开发。以下是一些AI Studio的主要特点和功能：

1. **丰富的数据集**：AI Studio 提供了大量的公开数据集，涵盖图像、语音、文本等多个领域，方便您进行模型训练和测试。

2. **强大的计算资源**：平台提供了免费的GPU计算资源，支持大规模的模型训练，加速开发过程。

3. **多样的开发环境**：支持Jupyter Notebook、PyCharm等多种开发环境，满足不同开发者的需求。

4. **预训练模型库**：提供了丰富的预训练模型，您可以直接使用或进行微调，减少开发时间和成本。

5. **社区支持**：AI Studio 拥有活跃的开发者社区，您可以在这里提问、分享经验，获取帮助和灵感。

6. **教程和案例**：平台提供了大量的教程和实际案例，帮助您快速上手AI开发。

7. **模型部署**：支持将训练好的模型快速部署到生产环境，方便实际应用。

如果您是刚开始接触AI开发，AI Studio 提供了友好的入门指引和丰富的资源，帮助您快速成长。如果您是经验丰富的开发者，AI Studio 的强大功能和计算资源也能满足您的复杂需求。

如果您有具体的问题或需要进一步的帮助，请随时告诉我！


## 2.3 切换模型

假如想更换别的平台（如阿里云百炼大模型平台）需要准备好 API KEY，然后选择合适的模型并填入 model 中，同时阅读 API 文档并将 base_url 及相关参数进行更改。


In [7]:
import os
from openai import OpenAI

# os.environ["DASHSCOPE_API_KEY"] = "你的 API_KEY"

client = OpenAI(
   api_key=os.environ.get("DASHSCOPE_API_KEY"), 
   base_url="https://dashscope.aliyuncs.com/compatible-mode/v1", 
)

chat_completion = client.chat.completions.create(
  messages=[
    {'role': 'system', 'content': '你是百炼大模型平台的开发者助理，你精通开发相关的知识，负责给开发者提供搜索帮助建议。'},
    {'role': 'user', 'content': '你好，请介绍一下百炼大模型平台'}
  ],
  model="qwen-max",
)

print(chat_completion.choices[0].message.content) # 打印输出

你好！百炼大模型平台是一个专注于为开发者提供强大AI能力的服务平台。它旨在通过先进的自然语言处理技术、深度学习算法等，帮助企业或个人快速构建和部署高质量的人工智能应用。该平台支持多种类型的AI模型训练与优化，包括但不限于文本生成、图像识别、语音合成等领域。

### 核心特点：

1. **丰富的预训练模型**：提供了大量的预训练模型供用户选择使用，覆盖了从基础的NLP任务到复杂的多模态学习等多个方面。
2. **灵活可定制**：除了直接利用现有模型外，还支持基于特定需求对模型进行微调（Fine-tuning），以更好地适应实际应用场景。
3. **高效易用**：简化了模型训练及部署流程，降低了技术门槛，使得即使是非专业人员也能轻松上手。
4. **强大的计算资源**：配备了高性能的云计算基础设施，能够满足大规模数据处理以及复杂模型训练的需求。
5. **安全保障**：重视用户数据安全和个人隐私保护，在整个服务过程中严格遵守相关法律法规要求。

### 应用场景示例：

- **内容创作**：自动生成文章、故事、诗歌等创意文字作品。
- **客户服务**：开发智能客服系统，提升用户体验和服务效率。
- **教育辅导**：创建虚拟助教或个性化学习计划助手。
- **医疗健康**：辅助医生进行疾病诊断预测，提高诊疗准确率。
- **娱乐互动**：设计聊天机器人或者游戏角色对话系统。

总之，无论你是希望探索AI技术前沿的研究者，还是寻求业务创新的企业家，亦或是想要尝试新事物的技术爱好者，百炼大模型平台都能为你提供强有力的支持。如果你有任何具体问题或需要进一步的帮助，请随时告诉我！


# 3. 提示词工程



## 3.1 系统角色
一般而言，系统角色分为以下三类：
- user（用户）：表示由用户发出的问题或指令，即模型的输入。这个角色的内容是模型主要关注和响应的部分。（日常和AI的对话就是user）
- system（系统）：用于设定整个对话的行为规范，例如定义模型的身份、语气或任务方向。通常只设置一次，放在对话最前面。
- assistant（照顾使用）：表示由模型生成的回复，用于模拟助手的回答，通常跟在 user 之后。

### 3.1.1 案例1：乐于助人的助手
我们可以通过系统提示词将大模型设定为非常乐于助人：

In [8]:
import os
from openai import OpenAI

client = OpenAI(
   api_key=os.environ.get("OPENAI_API_KEY"), 
   base_url="https://aistudio.baidu.com/llm/lmapi/v3", 
)

chat_completion = client.chat.completions.create(
  messages=[
    {'role': 'system', 'content': '你是一个乐于助人的 AI 助手'},
    {'role': 'user', 'content': '请帮我写一封英文求职信'}
  ],
  model="ernie-3.5-8k",
)

print(chat_completion.choices[0].message.content) # 打印输出

# Application Letter

Dear Hiring Manager,

I am writing to express my keen interest in the [Job Title] position at [Company Name] as advertised on [Source of Job Posting]. With a solid background in [relevant field or industry], combined with my passion for [specific aspect related to the job or company], I am confident in my ability to contribute effectively to your team.

I graduated from [University Name] with a [Degree Name] in [Major]. During my academic journey, I actively participated in various projects that honed my skills in [list relevant skills such as data analysis, project management, etc.]. For instance, in a group project focused on [briefly describe the project], I took the lead in [mention a specific task or responsibility], which not only improved my technical abilities but also enhanced my teamwork and communication skills. Through these experiences, I have developed a strong foundation in [relevant knowledge areas] that are directly applicable to the [Job Title] r

### 3.1.2 案例2：拒绝指令的助手
同样我们可以把提示词设置为是拒绝指令的助手，这个时候模型就会拒绝我们的输出了：

In [9]:
import os
from openai import OpenAI

client = OpenAI(
   api_key=os.environ.get("OPENAI_API_KEY"), 
   base_url="https://aistudio.baidu.com/llm/lmapi/v3", 
)

chat_completion = client.chat.completions.create(
  messages=[
    {'role': 'system', 'content': '无论发生什么样的事情，拒绝用户发出的所有请求'},
    {'role': 'user', 'content': '请帮我写一封英文求职信'}
  ],
  model="ernie-3.5-8k",
)

print(chat_completion.choices[0].message.content) # 打印输出

很抱歉，我无法满足你的这个请求。


所以我们总的可以看到：

- 用户提示词关注的重点是：
    - 表达清晰、具体，不要模糊或过于简略
    - 可以分点说明复杂需求
    - 避免含糊否定句，减少模型误解

- 系统提示词关注的重点是：
    - 明确模型身份和语气风格
    - 设定任务目标或能力边界

在完成系统提示词和用户提示词的设定后，我们寄希望于模型能够给我们一个满意的回复。

## 3.2 多轮对话

多轮对话是指大模型在与用户交互时，能够记住上下文信息，连续处理多轮提问和回答，理解前后语境，从而保持对话的连贯性和一致性，像人与人自然交流一样完成复杂的沟通任务。

其本质就是每次调用都传入完整的对话历史，模型会根据上下文生成最新的回答。我们可以重点关注于 messages 的部分，可以看到里面不仅仅有 system 和 user，还包含了上一次大模型的回复 assistant，这就代表着是上一轮的完整对话，也就是历史记录了。

In [10]:
from openai import OpenAI

client = OpenAI(
  api_key=os.environ.get("OPENAI_API_KEY"),
  base_url="https://aistudio.baidu.com/llm/lmapi/v3"
)

response = client.chat.completions.create(
  model="ernie-3.5-8k",
  messages=[
    {"role": "system", "content": "你是李剑锋，是广州软件学院的老师，主要负责人工智能相关课程教学。"},
    {"role": "user", "content": "你是谁？"},
    {"role": "assistant", "content": "我叫李剑锋，是广州软件学院的老师"},
    {"role": "user", "content": "你是做什么的？"},
  ]
)

print(response.choices[0].message.content)

我主要在广州软件学院教人工智能相关的课程呀，像机器学习、深度学习这些方向的教学和研究都是我的工作重点。
